# Course 5: Building a Small Bangla Language Model
## Session 9: Following instructions

**Goal:** build a chat template, fine-tune with the loss on answer tokens only, and set up the LoRA comparison.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every code cell here was run once on a CPU (November 2026) using the small built-in sample. Cells marked TODO or NOT RUN need your data, a GPU or the network. The LoRA cell was not run.

In [ ]:
SAMPLE_LINES = [
    "আমি বাংলায় গান গাই।",
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "বাংলা আমার মাতৃভাষা।",
    "আজ আকাশ খুব পরিষ্কার।",
    "ছোট মেয়েটি নদীর ধারে খেলছিল।",
    "বাবা বাজার থেকে মাছ কিনে আনলেন।",
    "মা রান্নাঘরে ভাত রান্না করছেন।",
    "আমরা সবাই মিলে স্কুলে যাই।",
    "বৃষ্টি হলে মাঠে পানি জমে।",
    "পাখিরা সকালে গান গায়।",
    "দাদু প্রতিদিন সন্ধ্যায় গল্প বলেন।",
    "ছেলেটি বই পড়তে খুব ভালোবাসে।",
    "নদীতে অনেক নৌকা চলছে।",
    "শীতকালে আমরা পিঠা খাই।",
    "ঢাকা বাংলাদেশের রাজধানী।",
    "আমার বন্ধু ভালো ফুটবল খেলে।",
]

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

def train_bpe(lines, vocab_size, specials=("[UNK]", "<|endoftext|>")):
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.NFC()
    tok.pre_tokenizer = pre_tokenizers.Metaspace()
    tok.decoder = decoders.Metaspace()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=list(specials))
    tok.train_from_iterator(lines, trainer)
    return tok

### 1. Tokenizer with chat special tokens

In [ ]:
SPECIALS = ("[UNK]", "<|endoftext|>", "<|user|>", "<|assistant|>", "<|end|>")
torch.manual_seed(0)
tok = train_bpe(SAMPLE_LINES * 20, 320, specials=SPECIALS)
USER, ASSISTANT, END = (tok.token_to_id(t) for t in ["<|user|>", "<|assistant|>", "<|end|>"])
print(USER, ASSISTANT, END)

### 2. Instruction examples
Self-made tasks from the corpus: complete the sentence. Replace with at least 2,000 examples and hold 200 out.

In [ ]:
pairs = []
for line in SAMPLE_LINES:
    words = line.split()
    if len(words) >= 3:
        pairs.append(("বাক্যটি শেষ করো: " + " ".join(words[:-1]), words[-1]))
train_pairs, held_out = pairs[:-3], pairs[-3:]
print(len(train_pairs), "training examples;", held_out[0])

### 3. Template and answer-only loss
Targets are -100 (ignored) everywhere except the answer and the end token.

In [ ]:
def build(question, answer=None):
    q = [USER] + tok.encode(question).ids + [ASSISTANT]
    if answer is None:
        return q
    a = tok.encode(answer).ids + [END]
    ids = q + a
    targets = [-100] * (len(q) - 1) + a          # predict a[0] from the ASSISTANT token onwards
    return ids[:-1], targets

x, y = build(*train_pairs[0])
print(len(x), len(y), y)

### 4. Fine-tune

In [ ]:
cfg = Config(vocab_size=tok.get_vocab_size(), block_size=48, n_layer=2, n_head=2, d_model=64)
model = GPT(cfg)          # TODO: load your pretrained checkpoint instead of starting fresh
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for epoch in range(60):
    for q, a in train_pairs:
        x, y = build(q, a)
        _, loss = model(torch.tensor([x]), torch.tensor([y]))
        opt.zero_grad(); loss.backward(); opt.step()
print("final loss:", round(loss.item(), 4))

### 5. Answer, stopping at the end token

In [ ]:
@torch.no_grad()
def answer(question, max_new=10):
    ids = torch.tensor([build(question)])
    out = []
    for _ in range(max_new):
        logits, _ = model(ids)
        nxt = logits[0, -1].argmax().item()
        if nxt == END:
            break
        out.append(nxt)
        ids = torch.cat([ids, torch.tensor([[nxt]])], dim=1)
    return tok.decode(out)

for q, a in train_pairs[:3]:
    print("seen    |", q, "->", answer(q), "| expected:", a)
for q, a in held_out:
    print("held out|", q, "->", answer(q), "| expected:", a)

The tiny model memorizes the seen examples and usually fails on the held-out ones. With real pretraining and thousands of examples, held-out answers improve; rate them blind.

### 6. LoRA on a small pretrained model (TODO, NOT RUN)
Needs a GPU and the network. Check the model's chat template and its license.

In [ ]:
# !pip install transformers peft
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import LoraConfig, get_peft_model
# name = "PUT-A-SMALL-MULTILINGUAL-MODEL-NAME-HERE"
# base = AutoModelForCausalLM.from_pretrained(name)
# lora = get_peft_model(base, LoraConfig(r=8, lora_alpha=16, target_modules="all-linear", task_type="CAUSAL_LM"))
# lora.print_trainable_parameters()